# **Multi-Asset Portfolio Performance, Risk & Attribution**

**FINM3422 — Assessment 2 - Group 8**

# **1.0 Introduction**

This report presents a performance, risk, and attribution analysis of a large Australian superannuation fund investing across five asset-class sleeves: Australian Equities (AUS EQ), International Equities (INTL EQ), Bonds / Fixed Income, Real Estate (RE), and Private Equity / Venture Capital (PE/VC). Each sleeve is managed by an external manager and evaluated against a designated benchmark index.

The analysis covers the 10-year period from January 2016 to December 2025 using monthly return data, assessed against a composite benchmark constructed from Strategic Asset Allocation (SAA) weights. Performance is evaluated at both sleeve and total fund level, with APRA-inspired supervisory checks applied to assess long-run return adequacy, volatility, drawdown resilience, and stress scenario behaviour. Active return is further decomposed using the Brinson attribution framework to identify the  contribution of allocation and selection decisions across all five sleeves.

# **2.0 Data Overview**

The dataset consists of monthly return series for five asset-class sleeves, 
covering both manager returns and corresponding benchmark indices over the 
period January 2016 to December 2025 (120 observations). Benchmarks reflect 
industry-standard proxies: the S&P/ASX 200 for AUS EQ, MSCI World 
ex-Australia for INTL EQ, a Bloomberg AusBond proxy for Bonds, an A-REIT 
proxy for Real Estate, and a stylised synthetic benchmark for PE/VC. A 
risk-free rate proxy is included for Sharpe ratio calculations.

All data is loaded via a centralised pipeline that enforces a DatetimeIndex, 
validates monthly frequency, confirms full manager-benchmark date alignment, 
and checks for missing values prior to analysis.

Two sets of weights are used throughout — the Tactical Asset Allocation (TAA), 
reflecting the fund's current active positioning, and the Strategic Asset 
Allocation (SAA), representing the long-term policy benchmark. Both are held 
constant across the analysis period:

| Asset Class            | TAA Weight | SAA Weight |
|------------------------|------------|------------|
| Australian Equities    | 35%        | 40%        |
| International Equities | 35%        | 30%        |
| Bonds                  | 15%        | 20%        |
| Real Estate            | 5%         | 5%         |
| PE/VC                  | 10%        | 5%         |

### 2.1 Environment Info, Imports and Confit

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

sys.path.append("../src")
from data_loader import load_data, validate_data
from performance import all_sleeves_summary, annualised_return, annualised_volatility, max_drawdown
from attribution import brinson_attribution

print(sys.version)
print("pandas:", pd.__version__, "numpy:", np.__version__)
pd.set_option("display.float_format", "{:.4f}".format)

### 2.2 Parameters and Paths

In [ ]:
# Base paths
BASE_path = "/Users/ameliaperry/Desktop/BAFE/FINM3422/python case/FINM3422-Group-8-Task-2-1/data/"

# TAA weights
taa_weights = pd.Series({
    "aus_eq": 0.35, 
    "intl_eq": 0.35, 
    "bonds": 0.15, 
    "re": 0.05, 
    "pevc": 0.10})

### 2.3 Data Load and Sanity Check

In [1]:
# Load and Validate Data
df_managers, df_benchmarks, df_rf, df_saa = load_data(BASE_path)
validate_data(df_managers, df_benchmarks, df_rf)
saa_weights = df_saa["Weight"]

# Sanity Checks
print("Any nulls in benchmarks?", df_benchmarks.isna().sum().sum())
print("Any nulls in managers?", df_managers.isna().sum().sum())
print("Dates aligned?", df_benchmarks.index.equals(df_managers.index))
print("Monotonic dates?", df_benchmarks.index.is_monotonic_increasing)
display(df_managers.head())
display(df_benchmarks.head())
display(df_rf.head()) 
display(df_saa.head())

NameError: name 'load_data' is not defined

# **3.0 Performance and Risk Analysis**

### 3.1 Methodology

Performance metrics are calculated using monthly returns and annualised 
geometrically to capture the compound growth rate. Together, these metrics 
provide a comprehensive view of both absolute and relative performance, 
covering return adequacy, risk efficiency, and downside resilience.

**Annualised Return** — geometric growth rate of monthly returns:
$$R_{\text{ann}} = \left(\prod_{t=1}^{n}(1 + r_t)\right)^{12/n} - 1$$
- $r_t$: monthly return in month $t$, $n$: number of monthly observations

**Annualised Volatility** — monthly standard deviation scaled annually:
$$\sigma_{\text{ann}} = \sigma_{\text{monthly}} \times \sqrt{12}$$

**Sharpe Ratio** — excess return above the risk-free rate per unit of total risk:
$$\text{Sharpe} = \frac{R_{\text{ann}} - R_{f,\text{ann}}}{\sigma_{\text{ann}}}$$
- $R_{f,\text{ann}}$: annualised risk-free rate

**Tracking Error** — annualised volatility of monthly active returns:
$$TE = \sigma(r_P - r_B) \times \sqrt{12}$$
- $r_P - r_B$: monthly active return (manager minus benchmark)

**Information Ratio** — active return per unit of active risk:
$$IR = \frac{R_P - R_B}{TE}$$

**Maximum Drawdown** — largest peak-to-trough decline in cumulative wealth:
$$MDD = \min_t \left(\frac{W_t - \max(W)}{\max(W)}\right)$$
- $W_t$: cumulative wealth at time $t$


### 3.2 Results Summary & Display

In [ ]:
summary = all_sleeves_summary(df_managers, df_benchmarks, df_rf)

pct_rows = [
    "Annualised Return (Manager)",
    "Annualised Return (Benchmark)",
    "Annualised Volatility",
    "Tracking Error",
    "Maximum Drawdown (Manager)",
    "Maximum Drawdown (Benchmark)",
]
ratio_rows = ["Sharpe Ratio", "Information Ratio"]

print("Table 1: Performance & Risk Summary (%)")
display(summary.loc[pct_rows].T.map(lambda x: f"{x:.2%}"))

print("Table 2: Ratios")
display(summary.loc[ratio_rows].T.map(lambda x: f"{x:.2f}"))

AMELIA:

Accross the five sleeves, manager returns were consistently above the relative benchmarks, suggesting overall positive active management over the full 10-year sample period. However, the level and consistency of outperformance varied significantly between sleeves. 

**Australian Equities** showed modest outperformance (6.91% vs 5.88%), with a low Sharpe index of 0.27, reflective of high volatility compared to return. Maximum drawdown of -24.63% is consistent with the significant Australian equity market fluctuations over this time. 

**International Equities** performed the strongest on both an absolute and risk-adjusted returns basis. The manager delivered 15.57% annualsied against a 13.94% benchmark while maintaining the lowest volatility of the equity sleeves at 11.96%. The Sharpe ratio of 1.05 is the only sleeve to achieve above 1.0, indicating genuine risk-adjusted perfromance. The low tracking error of 2.57% alongside an IR of 0.64 suggests the manager was disciplined and actve by consistently added value without taking large bets.

**Bonds** produced near-zero risk-adjusted returns, with the Sharpe Ratio of -0.00 reflecting that the return of 3.02% was only minorly above the risk-free rate over this period. This was to be expected, as the firs half of the sample period's low-yield environment compressed bond yields. The 2022 rate hiking cycle generated significant mark-to-market losses, reflectd in the -11.35 maximum drawdown. Despite this, the bonds manager produced the highest IR of 1.21 across all sleeves, outfperforming the benchmark by 1.13% with a tracking error of only 0.94%. Even though bonds achieved the lowest return, the manager for this sleeve was the most effective by adding substantial value relative to the benchmark despite the challenging rate environment.

**Real Estate** produced the second highest manager return at 10.84% value versus the benchmark of 8.03%, an outperformance of 2.81%. However, this also had the highest volatility of 18.72% and largest manager drawdown of -28.69% across akk sleeves. The tracking error almost three times larger than international equities of 7.10% indicates the manager took significant active risk to generate this return, also reflected in the relatively low IR of 0.39. The benchmarks lower drawdown of -37.41% suggests the manager navigated downside risk during periods of market stress, which is the primary source of outperformance. 

**PE/VC** delivered 10.24% annualised return with the lowest volatility of the sleeves at 10.86% and the second highest Sharpe ratio of 0.66. This smooth return profile is typical of private market assets, where valuations are reported infrequently and do not always accurately capture short-term market movements. The low tracking error of 2.08% and IR of 0.41 suggest steady, consistent outperformance rather than large active tilts.


### 3.3 Visualisations

In [ ]:
# Plots 1 to 5 - Cumulative Returns per sleeve
# manager vs benchmark
for sleeve in df_managers.columns:
    wealth_mgr = (1 + df_managers[sleeve]).cumprod()
    wealth_bm  = (1 + df_benchmarks[sleeve]).cumprod()
    plt.figure(figsize=(10, 4))
    plt.plot(wealth_mgr, label="Manager")
    plt.plot(wealth_bm, label="Benchmark")
    plt.title(sleeve.upper() + " Cumulative Returns")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True)
    plt.show()

# Plot 6 - Sharpe Ratios across all sleeves
sleeves = list(df_managers.columns)
sharpe_ratios = [summary.loc["Sharpe Ratio", sleeve] for sleeve in sleeves]

plt.figure(figsize=(8, 4))
plt.bar(sleeves, sharpe_ratios)
plt.title("Sharpe Ratios by Sleeve")
plt.ylabel("Sharpe Ratio")
plt.show()

#Plot 7 - Information Ratios across all sleeves
sleeves = list(df_managers.columns)
ir_ratios = [summary.loc["Information Ratio", sleeve] for sleeve in sleeves]

plt.figure(figsize=(8, 4))
plt.bar(sleeves, ir_ratios)
plt.title("Information Ratios by Sleeve")
plt.ylabel("Information Ratio")
plt.show()

AMELIA:

The cumulative return charts reinforce the summary table findings and provide further context on the timing and consistency of outperformance.

**Equity sleeves (AUS EQ & INTL EQ)** both show a sharp drawdown in early 2020 consistent with the COVID-19 shock, followed by a quicker recovery by the manager than the benchmark. INTL EQ shows the largest divergence, with the manager accelerating well above the benchmark from 2023 onwards to reach $4.20 versus $3.70 by 2026, the widest  terminal gap of all five sleeves and the primary driver of its 15.57% annualised return.

**Real Estate** experienced the most severe benchmark drawdown in 2020, falling back to approximately $1.00 while the manager held above $1.50. This justifies the large difference in maximum drawdown (-28.69% vs -37.41%). Both recovered strongly after 2021, with the manager sustaining a wide and growing gap through to 2026.

**Bonds and PE/VC** both show smooth, steady growth with the manager's returns consistently above the benchmark throughout the entire period and no visible severe drawdowns. The Bonds gap widens significantly after 2022, consistent with the manager navigating the rate hiking cycle more effectively than the benchmark.

**Sharpe and IR charts** confirm that International Equities and PE/VC are the strongest risk-adjusted performers on both measures, while AUS EQ is low on both. The Bonds IR standing well above all other sleeves despite its near-zero Sharpe highlights that absolute and relative performance metrics are highly relevant on the manager's skill.

MASON:

These graphs demonstrate the growth of $1 in the fund overtime through the lens of the manager's selections and allocations compared to the benchmark's. The most striking observations available to these graphical demonstrations are that the Real Estate and Bond sleeves had the largest divergence from the benchmark when adjusting for the scale of growth. In other areas like Australian Equity and PE/VC the performance is relatively consistent with the benchmarks, however there is signs of improvement in 2026 with PE/VC performance differentiating by approximately $0.50 (see Graph 5). Moreover, the International Equities graph suggests impressive performance as the manager consistently outperformed from inception, growing $1 to ~$4.25 vs the benchmark's ~$3.75 by 2026, which portrays a remarkable long-run compounding advantage (See Graph 2). 

The information ratio graph in figure 7 demonstrates that the Sharpe Ratio "standings" were likely dominated by return-driven bias. Bonds stood at the top, with an information ratio of 1.21. This is an exceptional result indicating the bonds manager generated active return with remarkable consistency relative to the benchmark risk taken. Other notable results include the International Equities ranking second, which is consistent with the Sharpe Ratio values (generating an information ratio around 0.64). Moreover, PE/VC and Real Estate information ratios of 0.41 and 0.40 respectively are broadly similar, both performing modestly. While both managers outperformed their benchmarks, the consistency of that outperformance was relatively limited, seen through the wider tracking errors relative to the active return generated.

# **4.0 APRA-Inspired Performance and Risk Checks**

### 4.1 Methodology

MASON:

APRA-inspired risk checks were applied to the total fund return, constructed using TAA weights and manager returns each month (since current compliance requires current figures). To complete this risk check, three simplified APRA-inspired checks were adopted, in particular Long-Run Return vs Objective (CPI + 3%), Annualised Volatility vs Risk Limit (12%), and Maximum Drawdown vs Threshold (-25%). To simulate the "bear" case results of the fund, two simulations were developed — an equity crash scenario (-20% applied to AUS EQ and INTL EQ returns) and a bond yield spike scenario (+150bps, implying approximately -6% on bonds assuming a duration of 4 years, with spillover effects across other asset classes). This test was made to observe the resilience of the fund in the given environments. 

The Australian Prudential Regulatory Authority (APRA) is the regulatory authority that oversees superfunds. It details official requirements that need to be met for the protection of superfund clients, if the superfunds do not meet these requirements they can often be screened more closely and clients receive an official letter providing a recommendation to switch superfun. This can often lead to runs on superfunds and heavily impacts their performance, so it is pivotal that superfund meet those requirments. Please find the particular APRA checks below used in the risk analysis:

1. Long-run return vs objective (CPI + 3%) - To ensure funds provide real value, funds must demonstrate they're beating a minimum return target over time, typically benchmarked as CPI + X% (e.g. CPI + 3%)
2. Annualised volatility vs risk limit - funds cannot take on excessive risk per APRA's checks, therefore they require a band of volatility. 12% is a defensible band and that is why it has been selected for the check. 
3. Maximum drawdown vs threshold - if the fund drops more than a certain amount from its peak (e.g. -25%), it signals a serious loss event that APRA would scrutinise. This ties into risk and risk management. 

### 4.2 TAA Weights and Total Fund Return

In [ ]:
#Total Fund Monthly Returns
# weighted average of manager returns using TAA weights
df_fund_monthly = df_managers.mul(taa_weights).sum(axis=1)

# Total Benchmark Monthly Returns
# weighted average of benchmark returns using SAA weights
saa_weights = df_saa["Weight"]
df_benchmark_monthly = df_benchmarks.mul(saa_weights, axis=1).sum(axis=1)

print("Fund monthly returns (first 5):")
display(df_fund_monthly.head())

### 4.3 APRA Checks

In [ ]:
# Inputs
CPI_TARGET = 0.025 # assumed CPI of 2.5%
RETURN_PREMIUM = 0.035 # difference between CPI and return target
RETURN_TARGET = CPI_TARGET + RETURN_PREMIUM # 6% return target
VOL_LIMIT = 0.10 # 10% annualised volatility limit
DRAWDOWN_LIMIT = -0.25 # -25% maximum drawdown limit

# Fund Level Metrics
fund_ann_return = annualised_return(df_fund_monthly)
fund_ann_vol = annualised_volatility(df_fund_monthly)
fund_mdd = max_drawdown(df_fund_monthly)

# Check 1: Long-term return vs target
check1_pass = fund_ann_return >= RETURN_TARGET

# Check 2: Max Drawdown vs -25% Limit
check2_pass = fund_mdd >= DRAWDOWN_LIMIT

# Check 3: Volatility vs 10% Limit
check3_pass = fund_ann_vol <= VOL_LIMIT

# Results
APRA_results = pd.DataFrame({
    "Check": ["Long-run return >= CPI + 3.5% (6.0%)", "Max Drawdown >= -25%", "Annualised Volatility <= 10%"],
    "Fund Value": [f"{fund_ann_return:.2%}", f"{fund_mdd:.2%}", f"{fund_ann_vol:.2%}"],
    "Threshold": [f"{RETURN_TARGET:.2%}",f"{DRAWDOWN_LIMIT:.2%}", f"{VOL_LIMIT:.2%}"],
    "Pass / Fail": ["Pass" if check1_pass else "Fail", "Pass" if check2_pass else "Fail", "Pass" if check3_pass else "Fail"]})

print("Table 3: APRA Performance & Risk Checks:")
display(APRA_results)

In [ ]:
from charts import plot_fund_vs_benchmark, plot_apra_drawdown_threshold

# Build total fund return series
taa = pd.Series({
    "aus_eq": 0.35, "intl_eq": 0.35,
    "bonds": 0.15, "re": 0.05, "pevc": 0.10
})
portfolio_ret = df_managers.mul(taa).sum(axis=1)

plot_apra_drawdown_threshold(portfolio_ret, threshold=-0.25)
print("Figure 8: Portfolio Cumulative Returns with APRA Drawdown Threshold")

AMELIA 

The fund passed all three APRA-inspired checks over the full sample period. The long-run return of 10.38% greatly exceed the CPI + 3.5% target of 6.00%, suggesting the fund delivered strong real returns well above what would be needed to support the business's retirement payment obligations. This margin of 4.38% above the target means the fund is not just relying on favourable short-term conditions. 

The maximum drawdown of -9.22% passed the -25% threshold easily. This is a relatively shallow drawdown for a growth-oriented multi-asset fund that included the COVID-19 market disruption and 2022 rate-hiking cycle. The diversifications across five asset classes contributed to containing the total fund's decline in these periods, particularly the inclusion of Bonds and PE/VC which experienced the smallest drawdowns.

Annualised volatility of 6.85% sits within the 8-12% band typical of a balanced superannuation fund, and below the 10% limit. This is lower than to be expected given the 70% combined equity weight, and reflects the cushioning effect of the Bonds and PE/VC sleeves on the overall portfolio risk through diversification. 

MASON: 

In analysis of the APRA-inspired checks it is apparent that out of the selected three, all checks were passed. 

For the Long-Run Return vs Objective (CPI + 3%) check the fund generated an annualised return of 10.38% against the 5.50% threshold, signifying a 4.88% exceeded margin. This is a comfortable margin indicating the fund is deliver returns well above the level of inflation rate in Austalia, suggesting that clients of the superfund are receiving real returns. This relieves the fund from regulatory scrutiny in this department. 

The Annualised Volatility vs Risk Limit (12%) check was also passed. The fund's annualised level of volatility of 6.85% maintains within the 12% threshold limit, with a substantial amount of leverage left within the check. This is a positive sign for the superfund and implies that the managers are not taking on excessive risk to generate the returns they have, in which the level of returns implied they are doing so proficiently. 

Finally, the Maximum Drawdown vs Threshold (-25%) test was also satisfied. The superfund's maximum drawdown of -9.22% is signifcantly lower than the industry-common value of -25% used for the check. Despite the Real Estate market's maximum drawdown of -28.69%, due to the effects of diversification across the five asset classes, this translated to a stable maximum drawdown - demonstrating the benefit of TAA weighting. 

Therefore, all APRA-inspired checks were satisified, further demonstrating the effective return generating and risk management skills of the managers.

GABBY:

The fund passes three of the four APRA-inspired checks. The long-run return of 10.38% comfortably exceeds the CPI + 4% target of 6.00%, confirming strong value creation over the sample period. The maximum drawdown of −9.22% remains well within the −25% supervisory threshold, demonstrating resilience during adverse market conditions. Volatility of 6.85% falls below the 8–12% balanced portfolio band, placing the fund in the defensive category, which reflects the diversification benefit of the multi-asset structure rather than a failure of risk management. Under a simulated equity shock of −20% (applied simultaneously to the AUS EQ and INTL EQ sleeves), the fund returns a negative monthly outcome (-14.60%), highlighting significant sensitivity to equity market downturns given the combined 70% equity allocation under TAA weights. This underscores the importance of monitoring equity concentration risk within the fund's tactical positioning.

### 4.4 Shock Scenario A

20% Equity Crash in one month

MASON:

The first scenario simulates a sudden and detrimental crash to the equity markets for a single month, particularly the Australian Equities and International Equities markets. This market collapse is only maintained for a month, where the others retain their usual returns, which as a whole the extent to which the crash was simulated to sustain was a 20% decline in the existing return of that month. This is consistent with other hyper-level collapses in history, like Covid-19 and The Global Financial Crisis (GFC), as to truly determine the viability of the fund in such adverse conditions. Significantly, AUS EQ and INTL EQ together comrprise 70% of the fund's TAA weight, which implies it is highly susceptible to changes in the equity markets. Therefore, a collapse in the equity markets could represent the largest risk event the fund could face, so determining how it manages through the conditions is important. 

The model below was constructed by implementing the equity market crash scenario, which was a -20% shock additively factored in the second last month of the dataset (choice is irrelevant). Then the return was computed via TAA weighting and impact was measured. 

In [ ]:
# Apply shock to returns in month recent months
shock_returns_a = df_managers.iloc[-1].copy() # use last month returns as base
shock_returns_a["aus_eq"] = -0.20 # apply -20% shock to Australian equity
shock_returns_a["intl_eq"] = -0.20 # apply -20% shock to International equity
# Bonds, RE, PEVC assumed unaffected in shock scenario (equities only)

# Calculate shocked fund return for that month
return_shock_a = (taa_weights * shock_returns_a).sum()

# Comparison to normal return
return_normal_a = (taa_weights * df_managers.iloc[-1]).sum()

print("Shock Scenario: Equity Crash (-20% AUS EQ & INTL EQ in one month)")
shock_scenario_a_results = pd.DataFrame({
    "Scenario": ["Normal", "Equity Crash Shock"],
    "Portfolio Return": [f"{return_normal_a:.2%}", f"{return_shock_a:.2%}"],
    "AUS EQ Return": [f"{df_managers.iloc[-1]['aus_eq']:.2%}", "-20.00%"],
    "INTL EQ Return": [f"{df_managers.iloc[-1]['intl_eq']:.2%}", "-20.00%"]})
display(shock_scenario_a_results)
print(f"\nShock Impact on Portfolio Returns: {return_shock_a - return_normal_a:.2%}")

AMELIA:

A simultaneous -20% monthly shock to both Australian and International Equities produces a portfolio return of -13.73% for that month, compared to a normal return of 1.27%, a 15.01% reduction. This is a direct consequence of the fund's 70% combined Equities TAA weight, that means equity market shocks have a proportionally large impact on the total fund. The result 
demonstrates that while the fund's diversification across bonds, real estate, and PE/VC provides some buffer, being predominantly equity-driven means the fund would experience significant short-term losses in a severe equity market disruption.

MASON:

Under the -20% equity crash scenario, the total fund return deteriorated to -14.80% for the month, representing an impact of -14.00% relative to the baseline. This is a substantial deterioration of value in the month and suggests that the fund is heavily reliant on equity markets. Under normal conditions the maximum drawdown of the fund was -9.22%, yet in the equity crash scenario is propels to below 14.0%, meaning a synchronised equity shock of this magnitude would overwhelm the defensive contribution of bonds, real estate and PE/VC due to their combined weight of only 30%. This may highlight a structure risk in the fund, being the overweighting towards equities and this should be considered when the team evaluates risk-management strategies. 

### 4.5 Shock Scenario B - Bond Yield Spike

bond yields increase 150bps

MASON:

Bond yield spikes is pivotal to asset portfolio risk-management as it typically has an effect on all sleeves simultaneously. The origin of the yield spike in this scenario was a mass selloff, causing the price to decline and thus the increase in yields (given the inverse relationship between bond price and yields). The reason the bond yield spike affects all sleeves in the portfolio is due to the relationship between asset pricing and the risk-free rate in financial markets, as financial assets are valued through the discounting of future cash flows. In this scenario the yield spike is assumed to be 150 basis points, which over an assumed duration of 4-years, results in a total price change of 6% (extreme case). This extreme level should provide further scope into the defensibility of the superfund, given a broad deduction in return performance. The following assumptions have also been made: 
1. Real Estate declines by 5% given the susceptibility of the housing market to rate changes. 
2. Australian and International Equities decline by 2% respectively. 
3. Private Equity / Venture Capital to decline by 2%. 


In [ ]:
# bonds: -(duration x 1.5%) = -5%
# REITS: -5%
# Equities: -2%
# PE/VC: -2%

shock_returns_b = df_managers.iloc[-1].copy()
shock_returns_b["aus_eq"] = -0.02
shock_returns_b["intl_eq"] = -0.02
shock_returns_b["bonds"] = -0.05
shock_returns_b["re"] = -0.05
shock_returns_b["pevc"] = -0.02

return_shock_b = (taa_weights * shock_returns_b).sum()
return_normal_b = (taa_weights * df_managers.iloc[-1]).sum()

print("Shock Scenario B: Bond Yield Spike")
shock_scenario_b_results = pd.DataFrame({
    "Scenario": ["Normal (last month)", "Bond Yield Spike"],
    "AUS EQ": [f"{df_managers.iloc[-1]['aus_eq']:.2%}", "-2.00%"],
    "INTL EQ": [f"{df_managers.iloc[-1]['intl_eq']:.2%}", "-2.00%"],
    "Bonds": [f"{df_managers.iloc[-1]['bonds']:.2%}", "-5.00%"],
    "RE": [f"{df_managers.iloc[-1]['re']:.2%}", "-5.00%"],
    "PEVC": [f"{df_managers.iloc[-1]['pevc']:.2%}", "-2.00%"],
    "Portfolio Return": [f"{return_normal_b:.2%}", f"{return_shock_b:.2%}"]})
display(shock_scenario_b_results)
print(f"\nShock Impact on Portfolio Returns: {return_shock_b - return_normal_b:.2%}")

AMELIA:

The bond yield spike scenario produces a portfolio return of -2.60% against a normal return of 1.27% , a 3.87% reduction. This is significantly less severe than the equity crash scenario, despite applying decreases in returns across all five asset classes. This occurs due to the TAA weighs in the rate-sensitive assets being relatively small (Bonds at 15% weight, and RE at 5%). Additionally, the equity shock in this scenario of -2% is much less sever than the -20% in Scenario A. The cross-asset nature fo the shock, where rising rates affect bonds, real estate, and equities simulataneously, illustrates the interconnected risk of a rate-driven selloff. However, the funds 20% fixed income allocation limits the total damage to returns. This is consistent with the low overall volatility in the fund of 6.85%. 

MASON:

Using the same baseline month, the results have been formed by simulating the bond yield spike scenario. From Table 5 it is evident that the superfund is far more resilient to a broad-based shock like the bond selloff, given the relatively minimal impact on the baseline of 2.75% (when compared to the equities crash). Under the +150bps rate shock, the total fund return deteriorated to -3.55%, which is still a significant effect especially considering the duration sleeves like Real Estate and PE / VC take to grow.  

Therefore, the bond yield spike scenario produced a moderate total impact of -2.98% on the fund, adjusting the monthly return to -3.77% in the superfund from a baseline of -0.80%. Scenario 2 is significantly less impactful than Scenario 1, developing an impact of just -2.75% compared to 14.00%. This demonstrates that though a broad collapse in the bond yield market may have an insidious effects on the valuation of the assets in the portfolio, the concentration risk in equities is still the largest risk the managers should be aware of. Although the -2.75% is still a painful shock, the results confirm that the fund's primary tail risk remains its equity concentration — a rate shock of 150bps, while painful, is substantially less damaging than a synchronised equity market collapse given the fund's 70% combined TAA weight in AUS EQ and INTL EQ. 

### 4.6 Total Fund Vs. Benchmark Comparison

In [ ]:
# Summary Comparison Table
fund_vol = annualised_volatility(df_fund_monthly)
bm_return = annualised_return(df_benchmark_monthly)
bm_vol = annualised_volatility(df_benchmark_monthly)
bm_mdd = max_drawdown(df_benchmark_monthly)

comparison = pd.DataFrame({
    "Metric": ["Annualised Return", "Annualised Volatility", "Maximum Drawdown"],
    "Total Fund (TAA)": [f"{fund_ann_return:.2%}", f"{fund_vol:.2%}", f"{fund_mdd:.2%}"],
    "Composite Benchmark (SAA)": [f"{bm_return:.2%}", f"{bm_vol:.2%}", f"{bm_mdd:.2%}"]
})
print("Total Fund vs Composite Benchmark Performance:")
display(comparison)

AMELIA:

The total fund (TAA-weighted) outperformed the composite benchmark (SAA-weighted) across all three metric. The fund delivered an annualised return of 10.38% for the benchmark, while simultaneously achieving lower volatility (6.85% vs. 6.97%) and a less severe maximum drawdown (-9.22 vs -11.90)

The favourable outcome of generating higher returns with lower risk reflects the combined benefit of overweighting high-performance sleeves (INTL EQ and PE/VC) and the positive selection effects generated by managers across all five sleeves. These two effects are analysed in Section 5. 

GABBY:

The total fund delivered an annualised return of 10.38%, comfortably exceeding the 6% target return and 8.24% benchmark, with a Sharpe ratio of 1.07 indicating strong risk-adjusted performance. Portfolio volatility of 6.85% sits within the defensive band, reflecting the diversification benefit of combining five asset classes under the TAA weighting scheme. The maximum drawdown of −9.22% remains well within the −25% supervisory threshold, demonstrating strong downside resilience over the sample period.

# **5.0 Multi-Asset Performance Attribution**

### 5.1 Formulas and Methodology

MASON:

Brinson-Style Attribution analysis was conducted to evaluate the managers' selection performance. It was performed monthly and then aggregated to the full analysis period averages. The composite benchmark return was constructed using SAA weights and monthly returns. Ultimately, the allocation effects were calculated using the following: $$(w_{P,i} - w_{B,i}) \times R_{B,i}$$ Selection effects: $$w_{B,i} \times (R_{P,i} - R_{B,i})$$ Then the total active return for each sleeve as the sum of its allocation and selection effects. 

The Brinson Attribution Model is based on active weights relative to the benchmark with returns generally split between allocation and security selection effects. It is used to demonstrate the allocation effects of weighting funds into specific sectors, relative to the market allocation. From this model analysis can be conducted to determine whether or not the sector allocation has been more/less effective than simply investing in the market, and whether or not the specific selection within those sectors are more efficient than the general market (if your allocation in the portfolio in the sectors provide less returns than the market's, relative to weight, poor selection allocation is exhibited).

Active return is comprised of allocation and selection effects following the Brinson Attribution framework. Throughout this analysis section the calculations will be derived from formulae like below. 

**Composite Benchmark Return:**

SAA weights × benchmark returns

$$R_{B,t} = \sum_{i} w_{B,i} \times R_{B,i,t}$$

**Total Fund Return:**

TAA weights × Manager Returns

$$R_{P,t} = \sum_{i=1}^{N} w_{P,i,t} \cdot R_{P,i,t}$$

**Allocation Effect:**

Finds value from over/underweighting asset class

$$A_i = (w_{P,i} - w_{B,i}) \cdot R_{B,i}$$

**Selection Effect:**

Finds whether managers outperformed benchmarks

$$S_i = w_{B,i} (R_{P,i} - R_{B,i})$$

Effects are computed monthly then summed to full-sample total.

### 5.2 Imports and Attribution

In [ ]:
# Check saa_weights index match df_managers columns
saa_weights = df_saa["Weight"]
saa_weights.index = saa_weights.index.astype(str).str.lower()

# Full Sample Attribution
df_attrib = brinson_attribution(df_managers, df_benchmarks, taa_weights, saa_weights)
print("Brinson Attribution (Full Sample):")
display(df_attrib.map(lambda x: f"{x:.2%}"))

The Brinson attribution reveals that both allocation and selection decisions contributed positively to the fund's 2.13% annualised active return, with much higher relative selection effects.

Selection effects totalled 12.15%, indicating that manager skill within each asset class was the primary driver of outperformance. Every sleeve generated a positive selection effect, supporting that all five external managers outperformed their respective benchmarks (after adjusting for benchmark weight). International Equities (4.42%) and Bonds (2.18%) had the higest selection contributions, reflecting consistent manager outperformance in these sleeves throughout the period.

Allocation effects totalled 7.39%, driven primarily by overweighting International Equities relative to SAA (35% TAA vs 30% SAA). This decision contributed 6.90% to the total allocation effect, as INTL EQ was the strongest performing benchmark over the period and overweighting this class added substantial value. PE/VC was the second-largest positive allocation contributor (4.82%), where again a sleeve with strong benchmark return was overweighted (5% to 10%)

The negative allocation effects in AUS EQ (-3.34%) and Bonds (-0.99%) reflect the cost of underweighting these sleeves relative to SAA. AUS EQ was underweighted by 5% (35% TAA vs 40% SAA) and still delivered positive benchmark returns, meaning the underweight decision reduced performance, whilst the manager still outperformed the benchmark through selection. This highlights how a manager can add value through skill (positive selection) while the allocation decision simultaneously detracts (negative allocation).

As the Real Estate weights were the same across both TAA and SAA, the allocation effect is 0.00%, and all of RE's 0.99% total contribution came from selection.

MASON:

In overview, a total return of 0.1631% per month was generated, or a a 1.96% annualised return above the composite benchmark. This is compiled into two effects, the allocation and selection effects, which warranted returns of 0.0616% and 0.1012% per month respectively. This means that approximately 62% of all portfolio returns came from the selection effect, which can be used as a proxy for manager skill, and the other 38% from the tactical weight tilts - the allocation effect. Moreover, every sleeve contributed at least a positive value, meaning there were no detractors anywhere in the portfolio, signalling manager skill and an effective weighting framework to generate returns. 

The allocation effect measures the value added from the TAA weights deviating from SAA weights. In terms of contributors, bonds contributed the largest allocation contribution of around 0.0259% - evident in Table 6 - despite only being a 15% weighting in the TAA framework. This means that the fund was underweight bonds relatve to SAA, which it was at 15% against 20%, in which this decision was value additive considering the benchmark underperformed by the allocation contribution, signalling appropriate decision making from managers. In terms of Internation Equities, they contributed 0.0233% from allocation, implying the increase weighting towards the sleeve was an effective choice. Similar results can be found in PE / VC and Australian Equities, generating monthly returns of 0.0061% and 0.0063%, yet both results are very small and do not demonstrate a significant impact to the fund's performance. Alternatively, the TAA and SAA weightings in the Real Estate market are the same, so the allocation effects can not be determined. 

The selection effect measures the managers' skills in outperforming their own sleeve benchmarks regardless of the weighting - purely off the basis of their choices. Australian Equities had the strongest selection effect relative to its size of around 0.0350%. This is a positive for the superfund and implies the manager is skilled in their stock selection. However, the Sharpe Ratio for the Australian Equities sleeve did imply they take on more active risk than ideal given that stock selection. Alternatively, International Equities contributed 0.0368% from selection, which is substantial. Combined with its strong allocation effect the International Equities sleeve is the most complete to active return across both dimensions. Bonds selection effect of approximately 0.0182%, which although modest, is meaningful given the low volatility and stable tracking erros in the sleeve. This demonstrates a high quality outcome for the portfolio and supports the manager's skill from this sleeve. Finally, Real Estate and PE / VC both contributed selection effects of arounf 0.0082% and 0.0030% respectively. These values are miniscule, primariliy driven by their small weights but the effects relative to the benchmark are still positive iimplying potential. 

Therefore, through a Brinson-style attribution table it is clear that the predominant driving force for returns in the superfund is the selection effect, or manager skills. Selection (1.21% annualised) outweighed allocation (0.74% annualised), indicating the fund's managers collectively added value through there proficiency in asset selection. Considering all effects, both allocation and selection were positive relative to the benchmark is a very positive sign for the superfund, indicating healthy strategic allocation and skillful managers. 

**Key Takeaways**
1. All the effects in the Figure 8 are positive. This means that in every sector the fund has been allocated to, it has outperformed the benchmark. 
2. Selection effect is higher than the allocation effect (0.1012% > 0.0616%). This implies that the managers are skilled in picking individual equities, and that the fund benefits more from their expertise than simply overweighting into the right sectors. 
3. The TAA weightings outperformed SAA weightings, visualised by positive allocation effects, this suggest allocating more weight in equities (40%) and bonds (20%) was more efficient for the entire fund. 

GABBY:

Attribution analysis reveals that International Equities was the primary driver of active outperformance, contributing positively through both allocation and selection effects, reflecting both the tactical decision to overweight the sleeve and strong manager performance within it. Australian Equities detracted through allocation (−0.0334) despite positive selection (0.0420), indicating that the underweight position relative to SAA cost the fund despite the manager outperforming its benchmark. The PE/VC sleeve contributed meaningfully through allocation (0.0482), driven by the TAA overweight from 5% to 10%, though selection added only marginally (0.0036). Bonds and Real Estate contributed positively through selection, suggesting manager skill within both sleeves, though their small weights limited the overall impact. In summary, allocation decisions were the dominant driver of active return, with selection providing a consistent but smaller positive contribution across most sleeves.


In [ ]:
# Bar Chart of Attribution
sleeves = [s for s in df_attrib.index if s != "total"]
alloc  = [df_attrib.loc[s, "Allocation Effect"] for s in sleeves]
select = [df_attrib.loc[s, "Selection Effect"]  for s in sleeves]

x = range(len(sleeves))
plt.figure(figsize=(10, 5))
plt.bar([i - 0.2 for i in x], alloc,  width=0.4, label="allocation effect", color="blue")
plt.bar([i + 0.2 for i in x], select, width=0.4, label="selection effect", color="orange")
plt.xticks(x, sleeves)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Brinson Attribution: Allocation vs Selection Effect by Sleeve")
plt.ylabel("Contribution to Active Return (full-sample sum)")
plt.legend()
plt.show()

The bar chart depicts the dominant selection effect across most sleeves, with International Equities and Bonds producing the largest selection bars. The allocation effect for International Equities is the highest bar effect across effects and sleeves, supporting it as the most impactful active manager decision in the portfolio. The negative allocation bars for AUS EQ and Bonds are more than offset the positiive allocation effect bars for both cases. PE/VC's allocation (4.82%) significantly exceeds selection (0.36%), suggesting the value added came primarily from the decision to overweight the asset class rather than from manager skill within it.

### 5.3 Verify Active Return

In [ ]:
# total active return = fund return - composite benmark return
total_fund_ann = annualised_return(df_fund_monthly)
total_bench_ann = annualised_return(df_benchmark_monthly)
active_return = total_fund_ann - total_bench_ann

print(f"Total Fund Annualised Return: {total_fund_ann:.4%}")
print(f"Composite Benchmark Annualised Return: {total_bench_ann:.4%}")
print(f"Total Active Return: {active_return:.4%}")
print(f"Attribution Total (sum of allocation & selection effects): {df_attrib.loc['Total', 'Total Attribution']:.4%}")

The independently computed active return of 2.13% differs from the attribution 
total of 19.54% as the attribution effects are computed as cumulative monthly sums rather than annualised figures. Both confirm that the fund generated significant positive active return over the 10-year sample period through a combination of allocation and selection decisions.

# **6.0 Conclusion**

### Key Insights

Over the January 2016 to December 2025 sample period, the fund delivered 10.38% annualised against a composite benchmark of 8.24%, generating 2.14% active return while simultaneously achieving lower volatility (6.85% vs 6.97%) and a shallower maximum drawdown (-9.22% vs -11.90%). The fund passed all three APRA-inspired checks, with the return target exceeded by over 4%, drawdown within the -25% threshold, and volatility consistent with a balanced superannuation mandate.

Manager selection was the dominant driver of active return, contributing 12.15% versus 7.39% from allocation decisions. All five managers generated positive selection effects, with the Bonds manager producing the highest IR of 1.21 despite near-zero absolute returns.

International Equities was the single largest contributor to overall fund  performance, adding value through both an overweight allocation decision (TAA 35% vs SAA 30%) and strong manager outperformance, generating a total attribution of 11.32%. The shock scenario analysis identified the 
fund's vulnerability to equity crashes ficen the 70% combined equity weight producing a -13.73% monthly return, significantly larger than the -2.60% impact of a bond yield spike.

 ### Limitations

- **Constant weights:** Fixed TAA and SAA weights throughout the period do not reflect the continuous adjustments that occur in practice 

- **Attribution methodology:** The cumulative monthly attribution total of 19.54% is not directly comparable to the annualised active return of 2.13% due to differences in compounding. A rigorous attribution would apply geometric linking to produce annualised-consistent figures.

- **Simplified APRA checks:** The approximations used in this model do not reproduce the requirements of APRA Standard.

- **No costs or fees:** All returns are gross do not factor transaction costs and management fees, which would reduce reported outperformance.



MASON:

Over the full sample period from 2016 to 2026, the multi-asset superfund generated a strong outperformance across each of the five asset sleeves, Australian Equities, International Equities, Bonds, Real Estate, and Private Equity / Venture Capital. The superfund exhibited effectiveness through both the tactical asset allocation framework and the managers' selections. Each sleeve outperformed the respective benchmarks on an absolute return basis, and through a Brinson attribution analysis it was confirmed that this outperformance applied to all sleeves. 

From a risk management perspective, the superfund passed all the APRA-inspired checks and ensured it was operating within reglations. The selected checks it passed specifically were an annualised return of 10.38% against a CPI+3% objective, volatility of 6.85% against a 12% limit, and a maximum drawdown of -9.22% against a -25% threshold. Alternatively, through shock scenario analysis it was determined that the fund may be susceptible due to equity concentration risk, with a combined weighting of 70% in the TAA framework, any large collapses in the equities markets will significantly hurt the suerfund with a simulated 14.00% decline in a single month.  

In summary, the superfund has functioned effectively over the analysis period, generating strong annualised returns with a sustainable risk-management strategy, driven by a sound tactical allocation framework. Moreover, the managers of the superfund have demonstrated that they are skillful in their positions, as the selection effect dominated the fund's returns - responsible for 62%. In terms of sleeves, International Equities and Bonds stood out through their strong returns and effective risk allocation, demonstrated through the International Equities sleeve's tremendous Sharpe Ratio of 1.05 and the Bonds sleeve's consistency. However, the high concentration in equities will threaten the fund's stability if a threat of a synchronised equity market collapse was to occur, and this type of allocation should be reviewed going forward.  

GABBY

### Key Insights

The analysis delivers five key findings for the CIO. International Equities were the standout performer across the fund, generating an annualised return of 15.57% and a Sharpe ratio of 1.05, and contributing the largest active return through both positive allocation and selection effects. The total fund achieved an annualised return of 10.38% with a Sharpe ratio of 1.07, comfortably exceeding the 6% target return while maintaining a maximum drawdown of just −9.22%. The TAA decision to overweight INTL EQ and PE/VC relative to SAA weights was the primary driver of active outperformance, with allocation effects dominating selection across most sleeves. Australian Equities remain the weakest sleeve; despite positive manager selection, the underweight TAA position detracted from overall active return, warranting a review of the tactical positioning. Finally, the fund's volatility of 6.85% sits in the defensive band rather than the balanced band, suggesting the current asset mix may be more conservative than intended under the SAA design and this warrants consideration in future rebalancing decisions.

### Limitations

Several limitations should be considered when interpreting these results:
1. Both TAA and SAA weights are assumed constant throughout the sample period — in practice, weights would be rebalanced periodically, and the fixed-weight assumption may overstate or understate actual active returns.
2. Transaction costs, management fees, and taxes are excluded from all return calculations, meaning reported performance figures represent gross returns that would be lower on a net basis. 
3. The risk-free rate proxy is a simplified monthly series and may not accurately reflect the actual short-term rate environment across the full sample period, introducing potential noise into Sharpe and Information Ratio calculations. 
4. The Brinson attribution model used here is a simplified two-factor decomposition that omits the interaction effect — in practice, a three-factor model would more precisely attribute the joint impact of allocation and selection decisions. 
5. The PE/VC benchmark is a stylised synthetic series that may not accurately reflect the illiquidity premium or valuation smoothing inherent in private market investments, potentially understating true risk for that sleeve. 
6. The APRA-inspired checks applied here are simplified educational proxies and do not constitute actual regulatory compliance assessment.


### Conclusion

This report demonstrates that the multi-asset fund delivered strong risk-adjusted performance over the sample period, generating an annualised return of 10.38% and a Sharpe ratio of 1.07, both materially exceeding the composite benchmark. The TAA decision to overweight International Equities and PE/VC relative to SAA weights was the primary driver of active outperformance, while manager selection added further value across most sleeves. The fund's conservative volatility profile and strong drawdown resilience are consistent with the risk management expectations of a large Australian superannuation fund. These findings confirm the fund is well-positioned relative to its long-term investment mandate, though continued monitoring of equity concentration risk and rebalancing frequency is recommended.
